# Memora NG: train the hybrid ViT-LSTM

Runtime > Change runtime type > **T4 GPU**, then run the cells top to bottom.

The notebook trains a Vision Transformer + bidirectional LSTM on ordered MRI slice series, evaluates it on held-out **subjects**, and exports the two files the web app needs: `memora_vit_lstm.onnx` and `model_card.json`.

## 1. Upload the project
Upload `memora-ng.zip` (the same zip you run the website from) when prompted.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # choose memora-ng.zip
name = next(iter(up))
zipfile.ZipFile(name).extractall('/content')
%cd /content/memora-ng
!pip -q install -r requirements-train.txt
import torch; print('GPU:', torch.cuda.is_available())

## 2. Get the data

**Option A (default): OASIS-1 slice export from Kaggle.** Open, free, and every file name carries the subject and slice number, so real ordered series can be rebuilt.

1. On kaggle.com go to Settings > API > *Create New Token*. A `kaggle.json` file downloads.
2. Run the cell and upload that file.

If Kaggle changes the dataset's folder names, the loader only needs class folders whose names contain *non*, *very mild*, *mild* or *moderate*.

In [ ]:
from google.colab import files
import os, json
up = files.upload()                       # choose kaggle.json
creds = json.loads(next(iter(up.values())))
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = creds['username'], creds['key']
!pip -q install kagglehub
import kagglehub
DATA = kagglehub.dataset_download('ninadaithal/imagesoasis')
print(DATA); !ls $DATA

**Option B: your own export (for example ADNI, once access is approved).** Arrange it like this and set `DATA` to the root folder:

```
root/CN/<subject>/slice_001.png ...
root/MCI/<subject>/...
root/AD/<subject>/...
```

In [ ]:
# DATA = '/content/drive/MyDrive/adni_slices'   # uncomment for option B
import sys; sys.path.insert(0, '.')
from collections import Counter
from training.dataset import discover
from memora.ai.labels import CLASS_NAMES
series = discover(DATA)
print(len(series), 'series from', len({s.subject for s in series}), 'subjects')
print(Counter(CLASS_NAMES[s.label] for s in series))
print('slices in first series:', len(series[0].files))
assert len(series) > 50, 'Too few series found. Check the folder layout.'

## 3. Train
Stage A trains the LSTM and classifier with the ViT frozen. Stage B fine-tunes the last four ViT blocks. The first epoch is slower because preprocessed slices are being cached. Expect roughly 30 to 60 minutes on a T4.

In [ ]:
!python training/train.py --data "$DATA" --out model_store --cache /content/cache --epochs-a 4 --epochs-b 14 --batch 8 --max-series-per-subject 2

## 4. Results for Chapter 4
These numbers come from test subjects the model never saw during training.

In [ ]:
import json
card = json.load(open('model_store/model_card.json'))
t = card['test']
for k in ('accuracy','precision_macro','recall_macro','f1_macro','roc_auc_ovr_macro'):
    print(f'{k:20s} {t[k]:.4f}' if t[k] is not None else f'{k:20s} n/a')
print('\nConfusion matrix (rows = actual, columns = predicted):', *card['labels'], sep='\n  ')
for row in t['confusion_matrix']: print(row)
print('\nPer class:'); [print(' ', n, m) for n, m in t['per_class'].items()]

In [ ]:
import numpy as np, matplotlib.pyplot as plt
cm = np.array(t['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cm, cmap='YlOrBr')
keys = ['CN', 'MCI', 'AD']
ax.set_xticks(range(3), keys); ax.set_yticks(range(3), keys)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix, test set')
for i in range(3):
    for j in range(3): ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout(); plt.savefig('model_store/confusion_matrix.png', dpi=200); plt.show()

## 5. Download the model
Put both files in the website's `model_store` folder and restart it. The orange *Demonstration engine* badge turns green.

In [ ]:
from google.colab import files
for f in ('memora_vit_lstm.onnx', 'model_card.json', 'confusion_matrix.png'):
    files.download('model_store/' + f)

## 6. Optional: ViT-only baseline for the hypothesis test
Your hypotheses claim the LSTM improves detection. This trains the same ViT without the LSTM (slice features are simply averaged) on the same split, so Chapter 4 can compare the two. It writes to a separate folder and does not touch the model above.

In [ ]:
!python training/train.py --data "$DATA" --out baseline_out --cache /content/cache --epochs-a 4 --epochs-b 14 --batch 8 --max-series-per-subject 2 --vit-only
import json
b = json.load(open('baseline_out/baseline_vit_only.json'))['test']; h = json.load(open('model_store/model_card.json'))['test']
for k in ('accuracy','precision_macro','recall_macro','f1_macro','roc_auc_ovr_macro'):
    print(f"{k:20s} ViT only {b[k]:.4f}   ViT + LSTM {h[k]:.4f}")